# **SAMUDRAGUPTA JEJUKAR - 23102C0064**


# Retail Sales Integration & Analysis in R

### LAB 3 — Multi-source retail analytics

This notebook builds a small end-to-end retail analytics pipeline in R:

**CSV + JSON + Excel → cleaning → joins → sales analysis → customer segments → SQLite → business insights**

The UCI Online Retail dataset is used as the source data. The official UCI description reports 541,909 transactions and eight original fields, including invoice number, stock code, quantity, date, price, customer ID and country.


## 1. Install and load the R packages

In [1]:

required <- c(
  "readr", "readxl", "jsonlite", "dplyr",
  "DBI", "RSQLite", "tidyr", "stringr"
)

missing <- required[!required %in% rownames(installed.packages())]
if (length(missing) > 0) {
  install.packages(missing, repos = "https://cloud.r-project.org")
}

library(readr)
library(readxl)
library(jsonlite)
library(dplyr)
library(DBI)
library(RSQLite)
library(tidyr)
library(stringr)

cat("Packages loaded successfully.\n")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Packages loaded successfully.


## 2. Prepare the three source files

In [2]:

uci_file <- "Online Retail.xlsx"
uci_url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

if (!file.exists("transactions.csv") ||
    !file.exists("products.json") ||
    !file.exists("customers.xlsx")) {

  if (!file.exists(uci_file)) {
    download.file(uci_url, destfile = uci_file, mode = "wb", quiet = FALSE)
  }

  raw_retail <- read_excel(uci_file)

  # Transaction-level source
  transactions <- raw_retail %>%
    transmute(
      InvoiceNo = as.character(InvoiceNo),
      StockCode = as.character(StockCode),
      CustomerID = as.character(CustomerID),
      Quantity = as.numeric(Quantity),
      InvoiceDate = as.character(InvoiceDate)
    )

  write_csv(transactions, "transactions.csv")

  # Product lookup
  products <- raw_retail %>%
    transmute(
      StockCode = as.character(StockCode),
      Description = as.character(Description),
      UnitPrice = as.numeric(UnitPrice)
    ) %>%
    distinct(StockCode, .keep_all = TRUE)

  write_json(products, "products.json", pretty = TRUE, na = "null", auto_unbox = TRUE)

  # Customer lookup
  customers <- raw_retail %>%
    transmute(
      CustomerID = as.character(CustomerID),
      Country = as.character(Country)
    ) %>%
    distinct(CustomerID, .keep_all = TRUE)

  writexl_available <- requireNamespace("writexl", quietly = TRUE)
  if (!writexl_available) {
    install.packages("writexl", repos = "https://cloud.r-project.org")
    library(writexl)
  }

  writexl::write_xlsx(customers, "customers.xlsx")
}

cat("Source files ready:\n")
print(c("transactions.csv", "products.json", "customers.xlsx"))


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



Source files ready:
[1] "transactions.csv" "products.json"    "customers.xlsx"  



## 3. Import the heterogeneous data

The three sources deliberately use different formats:

- CSV for transactions
- JSON for products
- Excel for customers


In [3]:

transactions_raw <- read_csv(
  "transactions.csv",
  show_col_types = FALSE,
  col_types = cols(
    InvoiceNo = col_character(),
    StockCode = col_character(),
    CustomerID = col_character(),
    Quantity = col_double(),
    InvoiceDate = col_character()
  )
)

products_raw <- fromJSON("products.json", flatten = TRUE) %>%
  as_tibble()

customers_raw <- read_excel("customers.xlsx") %>%
  mutate(CustomerID = as.character(CustomerID))

cat("Transactions:", nrow(transactions_raw), "rows\n")
cat("Products:", nrow(products_raw), "rows\n")
cat("Customers:", nrow(customers_raw), "rows\n")

glimpse(transactions_raw)


Transactions: 541909 rows
Products: 4070 rows
Customers: 4373 rows
Rows: 541,909
Columns: 5
$ InvoiceNo   <chr> "536365", "536365", "536365", "536365", "536365", "536365"…
$ StockCode   <chr> "85123A", "71053", "84406B", "84029G", "84029E", "22752", …
$ CustomerID  <chr> "17850", "17850", "17850", "17850", "17850", "17850", "178…
$ Quantity    <dbl> 6, 6, 8, 6, 6, 2, 6, 6, 6, 32, 6, 6, 8, 6, 6, 3, 2, 3, 3, …
$ InvoiceDate <chr> "2010-12-01 08:26:00", "2010-12-01 08:26:00", "2010-12-01 …


## 4. Clean and standardize the inputs

In [4]:

# Basic standardisation
transactions <- transactions_raw %>%
  mutate(
    InvoiceNo = str_trim(InvoiceNo),
    StockCode = str_trim(StockCode),
    CustomerID = str_trim(CustomerID),
    InvoiceDate = as.POSIXct(
      InvoiceDate,
      format = "%Y-%m-%d %H:%M:%S",
      tz = "UTC"
    )
  )

products <- products_raw %>%
  transmute(
    StockCode = str_trim(as.character(StockCode)),
    Description = str_squish(as.character(Description)),
    UnitPrice = as.numeric(UnitPrice)
  )

customers <- customers_raw %>%
  transmute(
    CustomerID = str_trim(as.character(CustomerID)),
    Country = str_squish(as.character(Country))
  )

# Cleaning decisions:
# 1. Remove duplicate transaction rows.
# 2. Drop rows without a usable customer ID because customer-level analysis
#    requires a customer key.
# 3. Exclude zero/negative quantities because they do not represent positive sales.
# 4. Exclude zero/negative prices because revenue should represent valid sales.
# 5. Exclude missing product/customer lookup keys before joining.

before_clean <- nrow(transactions)

transactions_clean <- transactions %>%
  distinct() %>%
  filter(
    !is.na(CustomerID), CustomerID != "",
    !is.na(Quantity), Quantity > 0,
    !is.na(InvoiceNo), InvoiceNo != "",
    !is.na(StockCode), StockCode != ""
  )

products_clean <- products %>%
  filter(
    !is.na(StockCode), StockCode != "",
    !is.na(UnitPrice), UnitPrice > 0
  ) %>%
  distinct(StockCode, .keep_all = TRUE)

customers_clean <- customers %>%
  filter(!is.na(CustomerID), CustomerID != "", !is.na(Country)) %>%
  distinct(CustomerID, .keep_all = TRUE)

cat("Transaction rows before cleaning:", before_clean, "\n")
cat("Transaction rows after cleaning:", nrow(transactions_clean), "\n")
cat("Removed transaction rows:", before_clean - nrow(transactions_clean), "\n")


Transaction rows before cleaning: 541909 
Transaction rows after cleaning: 392708 
Removed transaction rows: 149201 


## 5. Create Revenue

In [5]:

# Revenue = Quantity × UnitPrice
#
# UnitPrice is stored in the product lookup, so the calculation is completed
# after the product join.

sales_enriched <- transactions_clean %>%
  left_join(products_clean, by = "StockCode") %>%
  left_join(customers_clean, by = "CustomerID") %>%
  mutate(Revenue = Quantity * UnitPrice)

sales_enriched %>%
  summarise(
    rows = n(),
    missing_description = sum(is.na(Description)),
    missing_price = sum(is.na(UnitPrice)),
    missing_country = sum(is.na(Country))
  )


rows,missing_description,missing_price,missing_country
<int>,<int>,<int>,<int>
392708,4831,4831,0


## 6. Integrate the sources with joins

In [6]:

# left_join is used because transactions are the fact table.
# We want to retain every cleaned transaction and then inspect whether
# lookup information is missing.

integrated_sales <- transactions_clean %>%
  left_join(products_clean, by = "StockCode") %>%
  left_join(customers_clean, by = "CustomerID") %>%
  mutate(Revenue = Quantity * UnitPrice)

cat("Integrated dimensions:", nrow(integrated_sales), "rows x",
    ncol(integrated_sales), "columns\n")

unmatched_products <- integrated_sales %>%
  filter(is.na(Description) | is.na(UnitPrice)) %>%
  distinct(StockCode)

unmatched_customers <- integrated_sales %>%
  filter(is.na(Country)) %>%
  distinct(CustomerID)

cat("Unmatched product keys:", nrow(unmatched_products), "\n")
cat("Unmatched customer keys:", nrow(unmatched_customers), "\n")

# For final analysis, keep only fully matched sales records.
retail_sales <- integrated_sales %>%
  filter(!is.na(UnitPrice), !is.na(Country), !is.na(Revenue))

cat("Final integrated dataset:", nrow(retail_sales), "rows x",
    ncol(retail_sales), "columns\n")


Integrated dimensions: 392708 rows x 9 columns
Unmatched product keys: 78 
Unmatched customer keys: 0 
Final integrated dataset: 387877 rows x 9 columns


## 7. Overall sales performance

In [7]:

total_revenue <- retail_sales %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE))

total_revenue


TotalRevenue
<dbl>
10752840


## 8. Top five products

In [8]:

top_products <- retail_sales %>%
  group_by(StockCode, Description) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    QuantitySold = sum(Quantity, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)

top_products


StockCode,Description,Revenue,QuantitySold
<chr>,<chr>,<dbl>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
47566,PARTY BUNTING,142437.56,15283
22423,REGENCY CAKESTAND 3 TIER,135604.80,12384
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93745.65,36763
23166,MEDIUM CERAMIC TOP STORAGE JAR,81032.64,77916


## 9. Top five countries

In [9]:

top_countries <- retail_sales %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue)) %>%
  slice_head(n = 5)

top_countries


Country,Revenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


## 10. Customer purchase value and segmentation

In [10]:

customer_value <- retail_sales %>%
  group_by(CustomerID, Country) %>%
  summarise(
    PurchaseValue = sum(Revenue, na.rm = TRUE),
    Orders = n_distinct(InvoiceNo),
    .groups = "drop"
  ) %>%
  arrange(desc(PurchaseValue))

top_customers <- customer_value %>%
  slice_head(n = 5)

top_customers


CustomerID,Country,PurchaseValue,Orders
<chr>,<chr>,<dbl>,<int>
18102,United Kingdom,408760.0,60
14646,Netherlands,357531.1,74
17450,United Kingdom,186038.0,45
14911,EIRE,182690.5,201
16446,United Kingdom,168472.5,2


In [11]:

# Data-driven thresholds:
# Low     = below 25th percentile
# Medium  = 25th to below 50th percentile
# High    = 50th to below 75th percentile
# Premium = 75th percentile or above

q <- quantile(
  customer_value$PurchaseValue,
  probs = c(0.25, 0.50, 0.75),
  na.rm = TRUE
)

customer_segments <- customer_value %>%
  mutate(
    Segment = case_when(
      PurchaseValue < q[[1]] ~ "Low Value",
      PurchaseValue < q[[2]] ~ "Medium Value",
      PurchaseValue < q[[3]] ~ "High Value",
      TRUE ~ "Premium"
    )
  )

segment_summary <- customer_segments %>%
  count(Segment, name = "Customers") %>%
  arrange(factor(
    Segment,
    levels = c("Low Value", "Medium Value", "High Value", "Premium")
  ))

segment_summary


Segment,Customers
<chr>,<int>
Low Value,1085
Medium Value,1084
High Value,1085
Premium,1085


## 11. High-performing vs underperforming markets

In [12]:

market_summary <- retail_sales %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    Customers = n_distinct(CustomerID),
    Orders = n_distinct(InvoiceNo),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue))

high_market <- market_summary %>% slice_head(n = 1)
low_market <- market_summary %>% slice_tail(n = 1)

cat("High-performing market:\n")
print(high_market)

cat("\nUnderperforming market (lowest revenue in the cleaned data):\n")
print(low_market)


High-performing market:
# A tibble: 1 × 4
  Country         Revenue Customers Orders
  <chr>             <dbl>     <int>  <int>
1 United Kingdom 8861857.      3921  16635

Underperforming market (lowest revenue in the cleaned data):
# A tibble: 1 × 4
  Country      Revenue Customers Orders
  <chr>          <dbl>     <int>  <int>
1 Saudi Arabia    181.         1      1



### Interpretation

The high-performing market is the country with the largest total revenue after cleaning.
The underperforming market is the country with the smallest total revenue in the same
cleaned dataset. Revenue is used as the primary comparison because it directly measures
sales performance; customer and order counts are shown as supporting context.


## 12. Store the final dataset in SQLite

In [13]:

db_file <- "retail_analysis.sqlite"

if (file.exists(db_file)) {
  file.remove(db_file)
}

con <- dbConnect(SQLite(), dbname = db_file)

dbWriteTable(
  con,
  "retail_sales",
  retail_sales,
  overwrite = TRUE,
  row.names = FALSE
)

dbGetQuery(con, "SELECT COUNT(*) AS row_count FROM retail_sales")


row_count
<int>
387877


## 13. SQL query 1 — top five customers

In [14]:

sql_top_customers <- dbGetQuery(
  con,
  "
  SELECT
      CustomerID,
      Country,
      ROUND(SUM(Revenue), 2) AS TotalRevenue
  FROM retail_sales
  GROUP BY CustomerID, Country
  ORDER BY TotalRevenue DESC
  LIMIT 5
  "
)

sql_top_customers


CustomerID,Country,TotalRevenue
<chr>,<chr>,<dbl>
18102,United Kingdom,408760.0
14646,Netherlands,357531.1
17450,United Kingdom,186038.0
14911,EIRE,182690.5
16446,United Kingdom,168472.5


## 14. SQL query 2 — revenue by country

In [15]:

sql_country_revenue <- dbGetQuery(
  con,
  "
  SELECT
      Country,
      ROUND(SUM(Revenue), 2) AS TotalRevenue
  FROM retail_sales
  GROUP BY Country
  ORDER BY TotalRevenue DESC
  "
)

head(sql_country_revenue, 10)


,Country,TotalRevenue
,<chr>,<dbl>
1,United Kingdom,8861857.13
2,Netherlands,363884.48
3,EIRE,331660.17
4,Germany,263818.97
5,France,226975.60
6,Australia,173918.61
7,Spain,67426.09
8,Switzerland,66619.97
9,Japan,48600.22


## 15. Three business insights

In [16]:

top_country_name <- top_countries$Country[[1]]
top_country_value <- top_countries$Revenue[[1]]

top_product_name <- top_products$Description[[1]]
top_product_value <- top_products$Revenue[[1]]

top_customer_id <- top_customers$CustomerID[[1]]
top_customer_value <- top_customers$PurchaseValue[[1]]

cat(
  "Insight 1: ", top_country_name,
  " is the strongest market in the cleaned dataset, generating ",
  round(top_country_value, 2), " in revenue.\n\n",
  "Insight 2: The highest-revenue product is '", top_product_name,
  "', contributing ", round(top_product_value, 2),
  " in sales.\n\n",
  "Insight 3: Customer ", top_customer_id,
  " has the largest purchase value at ",
  round(top_customer_value, 2),
  ", which indicates a high-value customer worth retaining.\n",
  sep = ""
)


Insight 1: United Kingdom is the strongest market in the cleaned dataset, generating 8861857 in revenue.

Insight 2: The highest-revenue product is 'PAPER CRAFT , LITTLE BIRDIE', contributing 168469.6 in sales.

Insight 3: Customer 18102 has the largest purchase value at 408760, which indicates a high-value customer worth retaining.



## 16. Close the database connection and save outputs

The final SQLite database and the three source files are generated in the
current working directory. These are the files to keep with the notebook for
submission.

In [17]:

dbDisconnect(con)
cat("Database connection closed. Final database:", db_file, "\n")


Database connection closed. Final database: retail_analysis.sqlite 
